In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt
from sklearn.cluster import KMeans
import openai
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# --- Load data ---

In [ ]:
config = load_config("configs/config_cluster_singlecell.yaml")
config.data_name = "BZ9"
config.refresh_paths()


In [ ]:
# --- Load data ---
data_path = str(dataset_dir("starmap", config.data_name))
x_data_name = "data.csv"  
index_col = 0
adata = sc.read_csv(f"{data_path}/{x_data_name}", first_column_names=True)
celltype_data = pd.read_csv(f"{data_path}/celltype.csv", index_col=index_col)  # Assuming first column is index
celltype_data.columns = ["cell_type"] 
pos_data = pd.read_csv(f"{data_path}/pos.csv", index_col=index_col)
pos_data.columns = ['x', 'y']
domain_data = pd.read_csv(f"{data_path}/domain.csv", index_col=index_col)
domain_data.columns = [config.name_truth]
adata.obs = adata.obs.join([celltype_data, pos_data, domain_data])

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

pos_data = adata.obs[['x', 'y']]
celltype_data = adata.obs[['cell_type']]
domain_data = adata.obs[[config.name_truth]]

# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))


# cluster cells

In [ ]:
# use KMeans instead of KModes

km = KMeans(n_clusters=len(domain_data[config.name_truth].unique()), random_state=42)
clusters = km.fit_predict(neighbor_normalized_df)
cluster_centers = km.cluster_centers_
cluster_centers = pd.DataFrame(cluster_centers, columns=neighbor_normalized_df.columns)
adata.obs['kmeans'] = clusters.astype(str)

In [ ]:
cluster_centers

# plot initial clusters

In [ ]:
sc.pl.scatter(adata,x="x",y="y", color="kmeans", title =  "Kmeans cluster results")
print(adjusted_rand_score(adata.obs["kmeans"], adata.obs[config.name_truth]))


# prompt
Important !!!!!!!!! name of niche

In [ ]:
# manually decide: use "domain_mapping"
# the order of domain_mapping should matches the order of cluster_centers
domain_mapping = {2: "Layer 1", 3: "Layer 2/3", 0: "Layer 5", 1: "Layer 6"}
cell_names_mapping = {'Astro': 'Astrocytes',
 'Endo': 'Endothelial cells',
 'L5-1': 'Layer 5 pyramidal neuron subtype 1',
 'Lhx6': 'Lhx6-expressing interneurons',
 'NPY': 'Neuropeptide Y-expressing interneurons',
 'Oligo': 'Oligodendrocytes',
 'Reln': 'Reelin-expressing cells',
 'SST': 'Somatostatin-expressing interneurons',
 'Smc': 'Smooth muscle cells',
 'VIP': 'Vasoactive intestinal peptide-expressing interneurons',
 'eL2/3': 'Excitatory neuron layer 2/3',
 'eL5-2': 'Excitatory neuron layer 5 subtype 2',
 'eL5-3': 'Excitatory neuron layer 5 subtype 3',
 'eL6-1': 'Excitatory neuron layer 6 subtype 1',
 'eL6-2': 'Excitatory neuron layer 6 subtype 2'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping
config.pos_data = pos_data
config.celltype_data = celltype_data


In [ ]:
# calculate prototype
one_shot_df = cluster_centers
print(one_shot_df.index)

# change the index of one_shot_df to be the same as the domain_mapping
one_shot_df.index = one_shot_df.index.map(config.domain_mapping)
print(one_shot_df.index)
# generate Comparison-based Prompt
config.oneshot_prompt = prompt.CP_celltype(one_shot_df, config)

In [ ]:
x = [i for i in range(len(adata)) if adata.obs[config.name_truth].iloc[i] == 4][30:31]
print(config.oneshot_prompt + prompt.oneshot_celltype(neighbor_normalized_df, x, config))


# GPT

## generate json

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt.oneshot_celltype, batch_size = 5000, n_rows = 1)


## submit

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_BZ5_cluster.yaml > BZ5_cluster.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
batch_id = "batch_671ec212c1d08190945b7519d30f8962"
file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
save_name = f"response_{config.data_name}_1_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
output_file_name = f"{config.output_path}/{save_name}"
# Open the file in write mode and save the string
with open(output_file_name, 'w') as file:
    file.write(file_response.text)  


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['cluster_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.value_counts()

# Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)


In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.oneshot_celltype, n_rows=1, 
                                                column_name="cluster_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
gemini_results_df.loc["552", "cluster_gemini"] = "Layer 5"

In [ ]:
gemini_results_df.index = gemini_results_df.index.astype(str)

In [ ]:
# which index number of gemini_results_df.index == '1'
# [i for i in range(len(gemini_results_df)) if gemini_results_df.index[i] == '1']
gemini_results_df.value_counts()


## plot

In [ ]:

adata.obs = adata.obs.join(gemini_results_df)
adata.obs = adata.obs.join(gpt_results_df)
adata.obs['cluster_gemini'] = adata.obs['cluster_gemini'].fillna("unknown")
adata.obs['cluster_gpt4o_mini'] = adata.obs['cluster_gpt4o_mini'].fillna("unknown")
sc.pl.scatter(adata, x="x", y="y", color="cluster_gemini", title =  f"cluster_gemini")
sc.pl.scatter(adata, x="x", y="y", color="cluster_gpt4o_mini", title =  f"cluster_gpt4o_mini")


In [ ]:
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['cluster_gemini']))
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['cluster_gpt4o_mini']))



## save results

In [ ]:
# gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

In [ ]:
config.model_type